In [1]:
import numpy as np
import xarray as xr

In [ ]:
SWOT_NC_DIR = "./data/SWOT_L3_Expert_v3_0"

In [3]:
def med_only(ds):
    med_mask = (
        (ds["latitude"] > 30.0681) & (ds["latitude"] < 47.3764) & 
        (ds["longitude"] > -6.0326) & (ds["longitude"] < 42.355)
    )
    
    ds = ds.where(med_mask.compute(), drop=True)

    return ds


def preprocess(ds):
    filename = ds.encoding["source"].split("/")[-1]
    
    if "i_num_line" in ds.data_vars:
        ds = ds.drop_vars(["i_num_line"])
    if "i_num_pixel" in ds.data_vars:
        ds = ds.drop_vars(["i_num_pixel"])

    ds = ds.assign_coords(num_lines=np.arange(ds.sizes["num_lines"]), num_pixels=np.arange(ds.sizes["num_pixels"]))

    ds = med_only(ds)

    cycle_num = int(filename.split("_")[5])
    ds = ds.expand_dims({"cycle_num": [cycle_num]})

    ds = ds.reset_coords(["latitude", "longitude"])

    return ds


def remove_land_only_pixels(ds):
    sea_mask = (~np.isnan(ds["ssha_filtered"])).any(dim="cycle_num")

    min_lat, max_lat = ds["latitude"].where(sea_mask).min(), ds["latitude"].where(sea_mask).max()
    min_lon, max_lon = ds["longitude"].where(sea_mask).min(), ds["longitude"].where(sea_mask).max()
    
    ds = ds.where(
        (
            (ds["latitude"] >= min_lat) & (ds["latitude"] <= max_lat) &
            (ds["longitude"] >= min_lon) & (ds["longitude"] <= max_lon)
        ).compute(),
        drop=True
    )

    return ds


def cleanup_vars(ds):
    ds["time"] = ds.time.max(dim=["num_pixels"], skipna=True).compute()
    ds["latitude"] = ds.latitude.max(dim=["cycle_num"], skipna=True).compute()
    ds["longitude"] = ds.longitude.max(dim=["cycle_num"], skipna=True).compute()
    ds["cross_track_distance"] = ds.cross_track_distance.max(dim=["cycle_num", "num_lines"], skipna=True).compute()
    ds["mdt"] = ds.mdt.max(dim=["cycle_num"], skipna=True).compute()
    ds["mss"] = ds.mss.max(dim=["cycle_num"], skipna=True).compute()

    ds["ssh_filtered"] = ds["ssha_filtered"] + ds["mdt"]

    ds["time"] = ds.time.interpolate_na(
        dim="cycle_num", method="linear", fill_value="extrapolate"
    ).interpolate_na(
        dim="num_lines", method="linear", fill_value="extrapolate"
    )

    ds["latitude"] = ds.latitude.interpolate_na(
        dim="num_lines", method="linear", fill_value="extrapolate"
    ).interpolate_na(
        dim="num_pixels", method="linear", fill_value="extrapolate"
    )

    ds["longitude"] = ds.longitude.interpolate_na(
        dim="num_lines", method="linear", fill_value="extrapolate"
    ).interpolate_na(
        dim="num_pixels", method="linear", fill_value="extrapolate"
    )

    ds = ds.set_coords(["time", "latitude", "longitude"])
    ds = ds.transpose("cycle_num", "num_lines", "num_pixels")

    for var in ds.data_vars:
        if "chunks" in ds[var].encoding:
            del ds[var].encoding["chunks"]
    ds = ds.chunk({"cycle_num": 10, "num_lines": -1, "num_pixels": -1})

    return ds


def open_dataset(pass_num):
    ds = xr.open_mfdataset(
        f"{SWOT_NC_DIR}/SWOT_L3_LR_SSH_Expert_*_{pass_num}_*_*_v3.0.nc", 
        concat_dim="cycle_num",
        combine="nested",
        preprocess=preprocess
    )

    ds = remove_land_only_pixels(ds)
    ds = cleanup_vars(ds)

    return ds

In [4]:
pass_003_ds = open_dataset("003")
pass_016_ds = open_dataset("016")

/var/folders/xc/bksmt58x2nq8jshz2jbf9b_m0000gn/T/ipykernel_95101/3938861531.py:89: FutureWarning: In a future version of xarray the default value for data_vars will change from data_vars='all' to data_vars=None. This is likely to lead to different results when multiple datasets have matching variables with overlapping values. To opt in to new defaults and get rid of these warnings now use `set_options(use_new_combine_kwarg_defaults=True) or set data_vars explicitly.
  ds = xr.open_mfdataset(
/var/folders/xc/bksmt58x2nq8jshz2jbf9b_m0000gn/T/ipykernel_95101/3938861531.py:89: FutureWarning: In a future version of xarray the default value for data_vars will change from data_vars='all' to data_vars=None. This is likely to lead to different results when multiple datasets have matching variables with overlapping values. To opt in to new defaults and get rid of these warnings now use `set_options(use_new_combine_kwarg_defaults=True) or set data_vars explicitly.
  ds = xr.open_mfdataset(


In [ ]:
pass_003_ds.to_zarr("./data/SWOT_L3_Expert_v3_0_pass003.zarr", mode="w")

/Users/bertrava/miniforge3/envs/swot-cyclogeo/lib/python3.11/site-packages/zarr/api/asynchronous.py:247: ZarrUserWarning: Consolidated metadata is currently not part in the Zarr format 3 specification. It may not be supported by other zarr implementations and may change in the future.
  warnings.warn(


In [ ]:
pass_016_ds.to_zarr("./data/SWOT_L3_Expert_v3_0_pass016.zarr", mode="w")

/Users/bertrava/miniforge3/envs/swot-cyclogeo/lib/python3.11/site-packages/zarr/api/asynchronous.py:247: ZarrUserWarning: Consolidated metadata is currently not part in the Zarr format 3 specification. It may not be supported by other zarr implementations and may change in the future.
  warnings.warn(
